## Environment Setup and SDWA Configuration

This cell initializes the SageMaker Studio environment and defines all shared
configuration used throughout the SDWA ingestion pipeline. It sets the AWS
region, creates a boto3 session, defines S3 bucket paths for raw and processed
data, specifies the Athena/Glue database, and maps logical SDWA table names to
their corresponding raw CSV filenames.


In [1]:
# -------------------------------------------------------------
# Environment Setup: AWS region, S3 bucket paths, and session
# -------------------------------------------------------------

import time
import awswrangler as wr
import pandas as pd
import boto3

# Create a boto3 session bound to the correct AWS region.
region = "us-east-1"
session = boto3.Session(region_name=region)

# S3 bucket where raw CSVs and Parquet outputs live.
bucket = "aai540-group5-public-866792937762-us-east-1-an"

# S3 prefixes for raw CSVs and processed Parquet datasets.
raw_prefix = "sdwa/raw/"
parquet_prefix = "sdwa/parquet/"

# Glue/Athena database name used for table registration.
database = "sdwa_db"

# Logical table → raw CSV filename mapping.
sdwa_files = {
    "events_milestones": "SDWA_EVENTS_MILESTONES.csv",
    "violations": "SDWA_VIOLATIONS_ENFORCEMENT.csv",
    "pws": "SDWA_PUB_WATER_SYSTEMS.csv",
    "lcr": "SDWA_LCR_SAMPLES.csv",
    "facilities": "SDWA_FACILITIES.csv",
    "service_areas": "SDWA_SERVICE_AREAS.csv",
    "geographic_areas": "SDWA_GEOGRAPHIC_AREAS.csv",
    "ref_codes": "SDWA_REF_CODE_VALUES.csv",
    "ansi_areas": "SDWA_REF_ANSI_AREAS.csv",
    "pn_assoc": "SDWA_PN_VIOLATION_ASSOC.csv",
    "site_visits": "SDWA_SITE_VISITS.csv",
}


## 1. Utility: Determine S3 Object Size

This helper function queries S3 to retrieve the size (in MB) of a raw CSV file.
The ingestion pipeline uses this size to decide whether to load the file normally
or stream it in chunks, which is important for memory‑constrained environments
like SageMaker Studio.


In [2]:
# -------------------------------------------------------------
# Utility: Return S3 object size in MB
# -------------------------------------------------------------

def get_s3_size_mb(key: str) -> float:
    """
    Query S3 metadata to determine the size of a raw CSV file.
    Used to decide between normal vs. streaming ingestion.
    """
    s3 = session.client("s3")
    resp = s3.head_object(Bucket=bucket, Key=f"{raw_prefix}{key}")
    return resp["ContentLength"] / (1024 * 1024)


## Helper: Read CSV with All Columns as Strings

This function reads a CSV file from S3 and coerces every column to string.
SDWA datasets contain inconsistent typing, so forcing all columns to string
ensures schema stability and prevents mixed‑type inference issues.


In [3]:
# -------------------------------------------------------------
# Helper: Read CSV from S3 with all columns forced to STRING
# -------------------------------------------------------------

def _read_csv_all_string(key: str, nrows: int | None = None) -> pd.DataFrame:
    """
    Load a CSV from S3 and coerce every column to string.
    This avoids mixed-type inference issues common in SDWA data.
    """
    path = f"s3://{bucket}/{raw_prefix}{key}"

    df = wr.s3.read_csv(
        path,
        dtype=str,          # Force all columns to string
        low_memory=False,   # Avoid partial type inference
        nrows=nrows
    )

    # Ensure dtype consistency across chunks
    df = df.astype(str)
    return df


# -------------------------------------------------------------
# Helper: Write DataFrame to Parquet (all STRING columns)
# -------------------------------------------------------------

def _write_parquet_all_string(table_name: str, df: pd.DataFrame) -> None:
    """
    Write a DataFrame to S3 as a Parquet dataset.
    All columns remain STRING to ensure schema stability in Athena.
    """
    out_path = f"s3://{bucket}/{parquet_prefix}{table_name}/"
    print(f" - Writing Parquet to: {out_path}")

    wr.s3.to_parquet(
        df=df,
        path=out_path,
        dataset=True,
        mode="overwrite"
    )

    print(f" - Parquet write complete for '{table_name}'.")


## Normal Ingestion for Small and Medium SDWA CSV Files

This cell defines the ingestion path used for SDWA tables that comfortably fit
into memory. It reads the entire CSV from S3 in one operation, forces all
columns to STRING to maintain schema consistency across the pipeline, logs the
row count for traceability, and writes the resulting DataFrame to S3 as a
Parquet dataset. This is the simplest and fastest ingestion mode and is used
whenever the file size falls below the streaming threshold.


In [4]:
# -------------------------------------------------------------
# Normal ingestion path for smaller CSVs
# -------------------------------------------------------------

def ingest_normal_csv(table_name: str, filename: str) -> None:
    """
    Load a CSV fully into memory, convert to STRING columns,
    and write to Parquet.
    """
    path = f"s3://{bucket}/{raw_prefix}{filename}"
    print(f" - Reading CSV (normal): {path}")

    df = _read_csv_all_string(filename)
    print(f" - Loaded {len(df):,} rows for '{table_name}'.")

    _write_parquet_all_string(table_name, df)


## Streaming Ingestion for Large SDWA CSV Files

This cell defines the ingestion path used for SDWA tables that are too large to
fit into memory (multi‑hundred‑MB to multi‑GB files). Instead of loading the
entire CSV at once, the file is streamed from S3 in fixed‑size chunks. Each
chunk is converted to STRING columns for schema consistency and appended to a
Parquet dataset in S3. This approach prevents memory exhaustion in SageMaker
Studio while still producing a single unified Parquet table for downstream
Athena queries and ML workflows.


In [5]:
# -------------------------------------------------------------
# Streaming ingestion for large CSVs (chunked processing)
# -------------------------------------------------------------

def ingest_large_csv_streaming(table_name: str, filename: str, chunksize: int = 200_000):
    """
    Stream a large CSV from S3 in chunks to avoid memory pressure.
    Each chunk is coerced to STRING and appended to a Parquet dataset.
    """
    path = f"s3://{bucket}/{raw_prefix}{filename}"
    print(f" - Streaming CSV from: {path}")

    chunk_iter = wr.s3.read_csv(
        path,
        dtype=str,          # Force all columns to string
        low_memory=False,
        chunksize=chunksize
    )

    out_path = f"s3://{bucket}/{parquet_prefix}{table_name}/"
    print(f" - Writing Parquet chunks to: {out_path}")

    first = True
    total_rows = 0
    chunk_idx = 0

    for chunk in chunk_iter:
        chunk_idx += 1
        chunk = chunk.astype(str)  # Ensure dtype consistency
        rows = len(chunk)
        total_rows += rows

        mode = "overwrite" if first else "append"
        first = False

        wr.s3.to_parquet(
            df=chunk,
            path=out_path,
            dataset=True,
            mode=mode
        )

        print(f"[chunk {chunk_idx}] wrote {rows:,} rows (mode={mode}).")

    print(f" - Completed streaming ingest for '{table_name}' ({total_rows:,} total rows).")


## Ingestion Router: Select Normal or Streaming Mode and Execute Full Pipeline

This cell defines the routing logic that determines whether each SDWA table
should be ingested using the normal in‑memory method or the streaming
chunk‑based method. The decision is based on the file’s size in S3 relative to
a configurable threshold (default 500 MB). After defining the router, the cell
iterates over all SDWA tables listed in `sdwa_files` and ingests each one
accordingly. This provides a single, automated entry point for the entire
ingestion workflow.


In [6]:
# -------------------------------------------------------------
# Route ingestion based on file size
# -------------------------------------------------------------

def ingest_table(table_name: str, filename: str, threshold_mb: float = 500.0):
    """
    Decide whether to ingest a table normally or via streaming.
    Threshold defaults to 500 MB.
    """
    size = get_s3_size_mb(filename)

    print(f"\n=== Ingesting {table_name} ({filename}) ===")
    print(f"File size: {size:.2f} MB")

    if size > threshold_mb:
        print(f"Using STREAMING ingestion for {table_name}...")
        ingest_large_csv_streaming(table_name, filename)
    else:
        print(f"Using NORMAL ingestion for {table_name}...")
        ingest_normal_csv(table_name, filename)

    print(f"Completed ingestion for {table_name}.")


# -------------------------------------------------------------
# Run ingestion for all SDWA tables
# -------------------------------------------------------------

for key, filename in sdwa_files.items():
    ingest_table(key, filename)



=== Ingesting events_milestones (SDWA_EVENTS_MILESTONES.csv) ===
File size: 65.76 MB
Using NORMAL ingestion for events_milestones...
 - Reading CSV (normal): s3://aai540-group5-public-866792937762-us-east-1-an/sdwa/raw/SDWA_EVENTS_MILESTONES.csv


2026-06-03 19:39:15,397	WARNING services.py:2137 -- WARNING: The object store is using /tmp instead of /dev/shm because /dev/shm has only 1908383744 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=4.58gb' to 'docker run' (or add it to the run_options list in a Ray cluster config). Make sure to set this to more than 30% of available RAM.
2026-06-03 19:39:15,539	INFO worker.py:2007 -- Started a local Ray instance.
/opt/conda/lib/python3.12/site-packages/ray/_private/worker.py:2046: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


 - Loaded 360,370 rows for 'events_milestones'.
 - Writing Parquet to: s3://aai540-group5-public-866792937762-us-east-1-an/sdwa/parquet/events_milestones/
 - Parquet write complete for 'events_milestones'.
Completed ingestion for events_milestones.

=== Ingesting violations (SDWA_VIOLATIONS_ENFORCEMENT.csv) ===
File size: 3886.44 MB
Using STREAMING ingestion for violations...
 - Streaming CSV from: s3://aai540-group5-public-866792937762-us-east-1-an/sdwa/raw/SDWA_VIOLATIONS_ENFORCEMENT.csv
 - Writing Parquet chunks to: s3://aai540-group5-public-866792937762-us-east-1-an/sdwa/parquet/violations/
[chunk 1] wrote 200,000 rows (mode=overwrite).
[chunk 2] wrote 200,000 rows (mode=append).
[chunk 3] wrote 200,000 rows (mode=append).
[chunk 4] wrote 200,000 rows (mode=append).
[chunk 5] wrote 200,000 rows (mode=append).
[chunk 6] wrote 200,000 rows (mode=append).
[chunk 7] wrote 200,000 rows (mode=append).
[chunk 8] wrote 200,000 rows (mode=append).
[chunk 9] wrote 200,000 rows (mode=append).

## Register Parquet Datasets as Athena Tables (All Columns as STRING)

This cell defines a helper function that registers a Parquet dataset as an
Athena table in the AWS Glue Catalog. It selects a sample Parquet file to
infer the column names, maps every column to the STRING type to maintain
schema consistency across all SDWA tables, and then creates or overwrites
the Athena table. This ensures that downstream SQL queries operate on a
stable, predictable schema regardless of the original CSV typing.


In [7]:
# -------------------------------------------------------------
# Register Parquet datasets as Athena tables (all STRING schema)
# -------------------------------------------------------------

def safe_create_parquet_table(database: str, table: str, path: str) -> None:
    """
    Register a Parquet dataset as an Athena table.
    All columns are mapped to STRING to ensure schema stability.
    """
    start = time.time()
    print(f"\n[START] Registering Athena table '{table}'")
    print(f"S3 Path: {path}")

    objects = wr.s3.list_objects(path)
    if not objects:
        print(f"[ERROR] No Parquet files found under {path}")
        return

    sample_path = objects[0]
    print(f" - Using sample file: {sample_path}")

    sample = wr.s3.read_parquet(sample_path)

    # Map all columns to Athena STRING type
    columns_types = {col: "string" for col in sample.columns}
    print(f" - Columns detected: {len(columns_types)} (all coerced to STRING)")

    wr.catalog.create_parquet_table(
        database=database,
        table=table,
        path=path,
        columns_types=columns_types,
        mode="overwrite"
    )

    elapsed = time.time() - start
    print(f"[DONE] Table '{table}' registered successfully in {elapsed:.2f}s")


## Ensure Athena/Glue Database Exists (Idempotent Check)

This cell verifies that the configured Athena/Glue database is present in the
AWS Glue Catalog. If the database does not already exist, it is created.  
If it *does* exist, the cell simply reports that fact.  
Because this check is idempotent, it can be safely re‑run without affecting
any existing tables or metadata.


In [8]:
# -------------------------------------------------------------
# Ensure Glue/Athena database exists (idempotent)
# -------------------------------------------------------------
dbs = wr.catalog.databases()

if database not in dbs["Database"].values:
    print(f"Creating database '{database}' in Glue Catalog...")
    wr.catalog.create_database(name=database)
else:
    print(f"Database '{database}' already exists.")


Database 'sdwa_db' already exists.


## Athena Schema Sanity Check for All Registered SDWA Tables

This cell performs a lightweight validation step to ensure that each Parquet
dataset was ingested and registered correctly. For every SDWA table, it reads a
small sample (one chunk) from the Parquet dataset and prints the pandas dtypes.
Because the ingestion pipeline forces all columns to STRING, every column should
appear as `object`/`string` here. This confirms schema consistency across tables
and prevents downstream Athena query failures caused by mixed or unexpected
types.


In [9]:
# -------------------------------------------------------------
# Athena Sanity Check: Confirm Parquet schemas match expectations
# -------------------------------------------------------------
# This loop reads a small sample from each Parquet dataset to verify:
#   1. Columns exist as expected
#   2. All columns are STRING (as enforced during ingestion)
#   3. Athena/Glue will interpret the schema consistently
#
# This is critical in SageMaker Studio because schema mismatches
# can cause Athena query failures later in the ML pipeline.

for table_name in sdwa_files.keys():
    path = f"s3://{bucket}/{parquet_prefix}{table_name}/"
    print(f"\n=== Schema check for '{table_name}' ===")

    # Read only the first chunk to avoid loading large datasets into memory
    df_iter = wr.s3.read_parquet(path, dataset=True, chunked=True)
    df_sample = next(df_iter)

    # Display column names and dtypes (should all be object/string)
    print(df_sample.dtypes)



=== Schema check for 'events_milestones' ===
SUBMISSIONYEARQUARTER    string[python]
PWSID                    string[python]
EVENT_SCHEDULE_ID        string[python]
EVENT_END_DATE           string[python]
EVENT_ACTUAL_DATE        string[python]
EVENT_COMMENTS_TEXT      string[python]
EVENT_MILESTONE_CODE     string[python]
EVENT_REASON_CODE        string[python]
FIRST_REPORTED_DATE      string[python]
LAST_REPORTED_DATE       string[python]
dtype: object

=== Schema check for 'violations' ===
SUBMISSIONYEARQUARTER           string[python]
PWSID                           string[python]
VIOLATION_ID                    string[python]
FACILITY_ID                     string[python]
COMPL_PER_BEGIN_DATE            string[python]
COMPL_PER_END_DATE              string[python]
NON_COMPL_PER_BEGIN_DATE        string[python]
NON_COMPL_PER_END_DATE          string[python]
PWS_DEACTIVATION_DATE           string[python]
VIOLATION_CODE                  string[python]
VIOLATION_CATEGORY_CODE        

## Athena Workgroup and Output Configuration

This cell defines the Athena execution environment used for all SQL queries in
the notebook. It sets the workgroup (defaulting to `"primary"`) and specifies
the S3 location where Athena query results will be written. Printing these
values provides a quick sanity check to ensure queries will execute in the
expected context and store results in the correct output bucket.


In [10]:
# -------------------------------------------------------------
# Athena configuration
# -------------------------------------------------------------
athena_workgroup = "primary"  
athena_output = f"s3://{bucket}/athena/results/"

print(f"Athena workgroup: {athena_workgroup}")
print(f"Athena output:    {athena_output}")


Athena workgroup: primary
Athena output:    s3://aai540-group5-public-866792937762-us-east-1-an/athena/results/


## List All Tables Registered in the Athena Database

This cell performs a quick verification step by querying the AWS Glue Catalog
for all tables associated with the configured Athena database. Displaying the
database/table pairs confirms that each SDWA dataset was successfully registered
during the ingestion and catalog‑creation steps, and provides a convenient
reference for subsequent Athena queries.


In [11]:
# -------------------------------------------------------------
# Athena test: list tables in the database
# -------------------------------------------------------------
tables = wr.catalog.tables(database=database)
print(tables[["Database", "Table"]])


   Database                                        Table
0   sdwa_db                                   ansi_areas
1   sdwa_db                            events_milestones
2   sdwa_db                                   facilities
3   sdwa_db                             geographic_areas
4   sdwa_db                                          lcr
5   sdwa_db                                     pn_assoc
6   sdwa_db                                          pws
7   sdwa_db                              pws_master_prod
8   sdwa_db                              pws_master_test
9   sdwa_db                             pws_master_train
10  sdwa_db                               pws_master_val
11  sdwa_db                                    ref_codes
12  sdwa_db                                service_areas
13  sdwa_db                                  site_visits
14  sdwa_db  temp_table_24b7e38abcb34ee1afa1f23d19e99373
15  sdwa_db                                   violations


## Preview a Sample of Rows from Each Athena Table

This cell performs a quick validation step by querying Athena for the first five
rows of every SDWA table registered in the Glue Catalog. It confirms that each
table is readable, that the Parquet datasets were registered correctly, and that
Athena can successfully execute SQL queries against them. This provides an early
sanity check before performing more complex joins or feature engineering.


In [12]:
# -------------------------------------------------------------
# Athena test: sample a few rows from each table
# -------------------------------------------------------------
for table_name in sdwa_files.keys():
    print(f"\n=== Athena preview: {table_name} ===")
    query = f"SELECT * FROM {database}.{table_name} LIMIT 5"
    df_preview = wr.athena.read_sql_query(
        sql=query,
        database=database,
        workgroup=athena_workgroup,
        s3_output=athena_output,
    )
    print(df_preview)



=== Athena preview: events_milestones ===
  submissionyearquarter      pwsid event_schedule_id event_end_date  \
0                2026Q1  AL0001223     TTT-AL1813870     09/07/2022   
1                2026Q1  AL0001566     TTT-AL1816160     10/24/2022   
2                2026Q1  IN2492893      TTT-IN516540     10/19/2022   
3                2026Q1  IN2520141      TTT-IN544720     04/06/2023   
4                2026Q1  IN2860031      TTT-IN598640     08/22/2024   

  event_actual_date                                event_comments_text  \
0        10/06/2022  SUBMIT  LEVEL 1 TCR ASSESMENT;0;Sample Lab ID:...   
1        11/19/2022  SUBMIT  LEVEL 1 TCR ASSESMENT;0;Sample Lab ID:...   
2        11/03/2022  LVL2 TTT MULTIPLE TC+ 2ND LVL 1-12 MN;0;Sample...   
3        04/16/2023  LVL2 TTT TC+/EC- WO RPTS 2ND LVL 1-12 MN;0;Sam...   
4        09/15/2024  LVL2 TTT MULTIPLE TC+ 2ND LVL 1-12 MN;0;Sample...   

  event_milestone_code event_reason_code first_reported_date  \
0                 RTL

## Joinability Checks for Core SDWA Relational Keys

This cell defines a reusable helper function that measures how well two SDWA
tables can be joined on a shared key (typically `pwsid`). For each table pair,
it reports both the total number of rows in the left table and how many of those
successfully match a row in the right table. The cell then runs joinability
checks between PWS and three critical datasets—Violations, LCR Samples, and
Service Areas—to validate relational integrity before constructing the PWS
master feature table.


In [13]:
# -------------------------------------------------------------
# 14. Joinability checks for core keys
# -------------------------------------------------------------

def check_key_overlap(table1, key1, table2, key2):
    q = f"""
    SELECT
        COUNT(*) AS total,
        COUNT(CASE WHEN t2.{key2} IS NOT NULL THEN 1 END) AS matched
    FROM {database}.{table1} t1
    LEFT JOIN {database}.{table2} t2
        ON t1.{key1} = t2.{key2}
    """
    return wr.athena.read_sql_query(q, database, workgroup=athena_workgroup, s3_output=athena_output)

print("PWS <--> Violations")
display(check_key_overlap("pws", "pwsid", "violations", "pwsid"))

print("PWS <--> LCR")
display(check_key_overlap("pws", "pwsid", "lcr", "pwsid"))

print("PWS <--> Service Areas")
display(check_key_overlap("pws", "pwsid", "service_areas", "pwsid"))


PWS <--> Violations


,total,matched
0,15473495,15298031


PWS <--> LCR


,total,matched
0,1268571,924498


PWS <--> Service Areas


,total,matched
0,477670,422099


## Aggregate Violations by PWSID Using Athena

This cell computes violation‑level summary features for each Public Water System
(PWSID). It queries Athena to produce two key aggregates: the total number of
violations and the number of distinct violation IDs associated with each system.
These features are later joined into the PWS master table for modeling and
analysis.


In [14]:
# -------------------------------------------------------------
# Violations aggregated by PWSID
# -------------------------------------------------------------
viol_agg = wr.athena.read_sql_query(
    f"""
    SELECT
        pwsid,
        COUNT(*) AS violation_count,
        COUNT(DISTINCT violation_id) AS distinct_violations
    FROM {database}.violations
    GROUP BY pwsid
    """,
    database,
    workgroup=athena_workgroup,
    s3_output=athena_output
)


## LCR Sampling Aggregated by PWSID

This cell computes Lead and Copper Rule (LCR) sampling statistics for each
Public Water System (PWSID). It queries Athena to produce two key features:
the total number of LCR samples and the average measured sample value
(after cleaning placeholder values like `"."`, empty strings, and `"nan"`).
These aggregates are later joined into the unified PWS master feature table.


In [15]:
# -------------------------------------------------------------
# LCR sampling aggregated by PWSID
# -------------------------------------------------------------
lcr_agg = wr.athena.read_sql_query(
    f"""
    SELECT
        pwsid,
        COUNT(*) AS lcr_samples,
        AVG(CAST(NULLIF(sample_measure, '.') AS DOUBLE)) AS avg_lcr_result
    FROM {database}.lcr
    WHERE sample_measure IS NOT NULL
          AND sample_measure NOT IN ('', 'nan')
    GROUP BY pwsid
    """,
    database,
    workgroup=athena_workgroup,
    s3_output=athena_output
)



## Build the Unified PWS‑Level Master Feature Table

This cell constructs the full PWS‑level master dataset by joining Public Water
System metadata with four aggregated feature groups: violations, LCR sampling,
service area characteristics, and geographic attributes. Each aggregation is
computed in a CTE for clarity and then left‑joined to the PWS table to ensure
every system is retained, even if some feature groups have no matching records.
The resulting table serves as the foundation for all downstream modeling,
analysis, and validation steps in the SDWA pipeline.


In [16]:
pws_master = wr.athena.read_sql_query(
    f"""
    WITH viol_agg AS (
        SELECT
            pwsid,
            COUNT(*) AS violation_count,
            COUNT(*) FILTER (WHERE enforcement_action_type_code IS NOT NULL) AS enforcement_actions
        FROM {database}.violations
        GROUP BY pwsid
    ),

    lcr_agg AS (
        SELECT
            pwsid,
            COUNT(*) AS lcr_samples,
            AVG(CAST(NULLIF(sample_measure, '.') AS DOUBLE)) AS avg_lcr_result
        FROM {database}.lcr
        WHERE sample_measure IS NOT NULL
              AND sample_measure NOT IN ('', 'nan')
        GROUP BY pwsid
    ),

    service_agg AS (
        SELECT
            pwsid,
            COUNT(*) AS service_area_count,
            MAX(service_area_type_code) AS primary_service_area_type
        FROM {database}.service_areas
        GROUP BY pwsid
    ),

    geo_agg AS (
        SELECT
            pwsid,
            MAX(state_served) AS state,
            MAX(county_served) AS county,
            MAX(city_served) AS city,
            MAX(area_type_code) AS area_type
        FROM {database}.geographic_areas
        GROUP BY pwsid
    )

    SELECT
        pws.pwsid,
        pws.pws_name,
        pws.pws_type_code,
        pws.primacy_agency_code,
        pws.epa_region,
        pws.state_code,
        pws.source_water_protection_code,
        pws.seasonal_startup_system,

        -- Aggregated features
        v.violation_count,
        v.enforcement_actions,
        l.lcr_samples,
        l.avg_lcr_result,
        s.service_area_count,
        s.primary_service_area_type,
        g.state,
        g.county,
        g.city,
        g.area_type

    FROM {database}.pws pws
    LEFT JOIN viol_agg v ON pws.pwsid = v.pwsid
    LEFT JOIN lcr_agg l ON pws.pwsid = l.pwsid
    LEFT JOIN service_agg s ON pws.pwsid = s.pwsid
    LEFT JOIN geo_agg g ON pws.pwsid = g.pwsid
    """,
    database,
    workgroup=athena_workgroup,
    s3_output=athena_output
)


## Inspect the PWS Master Table Shape and Descriptive Statistics

This cell prints the overall dimensions of the `pws_master` DataFrame and
generates a full descriptive summary (including categorical columns) to verify
that the unified feature table was constructed correctly. This provides an early
sanity check on row counts, missingness patterns, and value distributions before
moving into modeling or further feature engineering.


In [17]:
print("pws_master shape:", pws_master.shape)
pws_master.describe(include="all").transpose()


pws_master shape: (433698, 18)


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
pwsid,433698,433698,WI4710292,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
pws_name,433698,391511,nan,896,NaN,NaN,NaN,NaN,NaN,NaN,NaN
pws_type_code,433698,4,TNCWS,287188,NaN,NaN,NaN,NaN,NaN,NaN,NaN
primacy_agency_code,433698,66,MI,27262,NaN,NaN,NaN,NaN,NaN,NaN,NaN
epa_region,433698,10,05,127302,NaN,NaN,NaN,NaN,NaN,NaN,NaN
state_code,433698,65,MI,26703,NaN,NaN,NaN,NaN,NaN,NaN,NaN
source_water_protection_code,433698,3,nan,314784,NaN,NaN,NaN,NaN,NaN,NaN,NaN
seasonal_startup_system,433698,6,nan,416892,NaN,NaN,NaN,NaN,NaN,NaN,NaN
violation_count,258234.0,<NA>,<NA>,<NA>,59.240964,178.817539,1.0,6.0,17.0,50.0,15272.0
enforcement_actions,258234.0,<NA>,<NA>,<NA>,59.240964,178.817539,1.0,6.0,17.0,50.0,15272.0


## Clean and Normalize the PWS Master Table (Mixed‑Type Safe)

This cell finalizes the construction of the `pws_master` feature table by
performing safe, dtype‑aware cleanup. It removes duplicate rows, converts all
numeric‑like columns to proper numeric types using `pd.to_numeric` with
`errors="coerce"` (ensuring invalid values become `NaN` instead of raising
errors), and fills missing values only in object/string columns. This preserves
numeric integrity for modeling while ensuring categorical fields contain no
nulls.


In [18]:
# -------------------------------------------------------------
# Clean up PWS master table (safe for mixed dtypes)
# -------------------------------------------------------------

# Drop duplicates
pws_master = pws_master.drop_duplicates()

# Identify numeric columns
numeric_cols = [
    "violation_count",
    "enforcement_actions",
    "lcr_samples",
    "avg_lcr_result",
    "service_area_count"
]

# Convert numeric-like strings to numeric first
for col in numeric_cols:
    if col in pws_master.columns:
        pws_master[col] = pd.to_numeric(pws_master[col], errors="coerce")

# Fill NA only in object/string columns
string_cols = pws_master.select_dtypes(include=["object"]).columns
pws_master[string_cols] = pws_master[string_cols].fillna("")


## Train/Val/Test/Prod Split (40/10/10/40)

This cell shuffles the fully‑cleaned `pws_master` feature table and partitions it
into four disjoint subsets:

- **40%** training  
- **10%** validation  
- **10%** test  
- **40%** production holdout (never used during model development)

The split is deterministic via `random_state=42`, ensuring reproducibility across
runs and environments. These partitions feed directly into the modeling and
evaluation pipeline.


In [19]:
# -------------------------------------------------------------
# 40% train, 10% val, 10% test, 40% prod
# -------------------------------------------------------------
pws_master = pws_master.sample(frac=1, random_state=42)

n = len(pws_master)
train_end = int(0.4 * n)
val_end   = int(0.5 * n)
test_end  = int(0.6 * n)

train = pws_master.iloc[:train_end]
val   = pws_master.iloc[train_end:val_end]
test  = pws_master.iloc[val_end:test_end]
prod  = pws_master.iloc[test_end:]

print(len(train), len(val), len(test), len(prod))


173479 43370 43369 173480


## Write PWS Master Splits to S3 as Parquet

This cell saves each of the four PWS‑level splits—train, validation, test, and
production—to S3 as Parquet datasets. Each split is written to its own prefix
under `pws_master/splits/`, using `mode="overwrite"` to ensure idempotent,
re‑runnable behavior. These Parquet outputs become the canonical, versionable
artifacts for downstream modeling, evaluation, and deployment workflows.


In [20]:
# -------------------------------------------------------------
# Write PWS master splits to S3 as Parquet
# -------------------------------------------------------------
def write_split(name, df):
    path = f"s3://{bucket}/pws_master/splits/{name}/"
    print(f"Writing {name} → {path}")
    wr.s3.to_parquet(
        df=df,
        path=path,
        dataset=True,
        mode="overwrite"
    )

write_split("train", train)
write_split("val", val)
write_split("test", test)
write_split("prod", prod)


Writing train → s3://aai540-group5-public-866792937762-us-east-1-an/pws_master/splits/train/
Writing val → s3://aai540-group5-public-866792937762-us-east-1-an/pws_master/splits/val/
Writing test → s3://aai540-group5-public-866792937762-us-east-1-an/pws_master/splits/test/
Writing prod → s3://aai540-group5-public-866792937762-us-east-1-an/pws_master/splits/prod/


## Register PWS Master Split Tables in Athena

This cell finalizes the split‑registration workflow by looping over the four
PWS master subsets—**train**, **val**, **test**, and **prod**—and registering
each one as an Athena table.  
Each split is written to its own S3 prefix under `pws_master/splits/`, and
`safe_create_parquet_table` ensures:

- schema inference from a sample Parquet file  
- all columns coerced to **STRING** for Glue/Athena stability  
- idempotent overwrite behavior  

These Athena tables become the queryable interfaces for validation, feature
inspection, and downstream ML workflows.


In [21]:
# -------------------------------------------------------------
# Register PWS master split tables in Athena
# -------------------------------------------------------------
for split_name in ["train", "val", "test", "prod"]:
    path = f"s3://{bucket}/pws_master/splits/{split_name}/"
    table_name = f"pws_master_{split_name}"
    print(f"Registering Athena table: {table_name}")
    safe_create_parquet_table(database, table_name, path)


Registering Athena table: pws_master_train

[START] Registering Athena table 'pws_master_train'
S3 Path: s3://aai540-group5-public-866792937762-us-east-1-an/pws_master/splits/train/
 - Using sample file: s3://aai540-group5-public-866792937762-us-east-1-an/pws_master/splits/train/5012dfa285f64dddbad56ede35d9a754.snappy.parquet
 - Columns detected: 18 (all coerced to STRING)
[DONE] Table 'pws_master_train' registered successfully in 2.83s
Registering Athena table: pws_master_val

[START] Registering Athena table 'pws_master_val'
S3 Path: s3://aai540-group5-public-866792937762-us-east-1-an/pws_master/splits/val/
 - Using sample file: s3://aai540-group5-public-866792937762-us-east-1-an/pws_master/splits/val/0933a83e0b8c47b58d8cc8ca943ef3f9.snappy.parquet
 - Columns detected: 18 (all coerced to STRING)
[DONE] Table 'pws_master_val' registered successfully in 1.68s
Registering Athena table: pws_master_test

[START] Registering Athena table 'pws_master_test'
S3 Path: s3://aai540-group5-public

## Verify Row Counts for All PWS Master Splits

This final validation step confirms that each PWS master split—**train**, **val**,  
**test**, and **prod**—was successfully written to S3 *and* correctly registered  
as an Athena table. For each split, the cell issues a simple `COUNT(*)` query  
against the corresponding Athena table and prints the resulting row count.

Matching counts between the in‑memory DataFrame partitions and the Athena tables  
ensures the split pipeline is lossless, deterministic, and ready for downstream  
ML workflows.


In [22]:
# -------------------------------------------------------------
# Verify counts match for PWS master splits
# -------------------------------------------------------------
for split in ["train", "val", "test", "prod"]:
    q = f"SELECT COUNT(*) AS rows FROM {database}.pws_master_{split}"
    result = wr.athena.read_sql_query(
        q,
        database,
        workgroup=athena_workgroup,
        s3_output=athena_output
    )
    print(split, result["rows"][0])


train 173479
val 43370
test 43369
prod 173480
